# 02 — Pré-processamento (CGMacros)
**Integrante 1 — Pré-processamento e análise dos dados**

Segue o roteiro Data-Centric da aula, adaptado a séries temporais, e usa as funções de `preprocessing.py`:

| Etapa | Na aula (housing) | Aqui |
|---|---|---|
| Limpar dados | imputar `total_bedrooms` | interpolar só buracos curtos de glicose/HR; descartar glicose impossível; duplicados |
| Novos atributos | `rooms_per_household` | features diárias: macros por refeição, horários, atividade, TIR… |
| Dois sensores | — | média alinhada dos dois (`consensus`): cancela o viés de cada sensor (Libre lê baixo, Dexcom lê alto) |
| Categóricos | one-hot | gênero e etnia |
| Teste separado | `train_test_split` estratificado | divisão **por participante**, estratificada por grupo (A1c) |
| Escala | `StandardScaler` / `MinMaxScaler` | ajustado **somente no treino** |
| Entrada × avaliação | — | D1–D6 (entrada) × D7–D9 (avaliação), alvo = glicose em D9 |

## 0. Configuração

In [ ]:
import os, sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append("..")
import preprocessing as P

RAW_DIR = Path(os.environ.get("CGMACROS_RAW", "../data/raw"))
OUT_DIR = Path(os.environ.get("CGMACROS_OUT", "../data/processed"))

# Decisões de projeto (justifique no relatório com base no notebook 01)
GLUCOSE_SOURCE = "consensus"  # 'consensus' (padrão) | 'libre_filled' | 'mean' | 'libre' | 'dexcom'  (ver notebook 01, seção 10)
DAY_MODE = "first_full"      # 'first_full': D1 = 1o dia com glicose suficiente (D1-D9 completos) | 'calendar' | 'from_start'
MIN_COVERAGE = 0.8           # fração mínima do dia com glicose para o dia contar como "completo"
SEED = 42

pd.set_option("display.max_columns", 40, "display.width", 160)
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True})

## 1. Limpeza de um participante — antes × depois
`clean_participant` faz: arredondar timestamps; juntar duplicados; grade regular de 1 min; glicose fora de 40–400 → NaN;
interpolar só buracos de até 30 min (glicose) / 10 min (HR, METs); unificar Libre+Dexcom (`consensus`); METs ÷ 10 (quem só tem `Intensity` tem os METs estimados pelas calorias); unificar lanches ('snack/snacks/snack 1'); macros absurdos (erros de digitação) → imputados;
nutrientes sem refeição = 0; numerar D1…Dn (D1 = primeiro dia completo).

In [ ]:
files = P.find_participant_files(RAW_DIR)
pid = sorted(files)[0]
raw = P.load_participant_csv(files[pid])
clean, stats = P.clean_participant(raw, pid, glucose_source=GLUCOSE_SOURCE, day_mode=DAY_MODE, min_coverage=MIN_COVERAGE)
print(pd.Series(stats).to_string())

In [ ]:
# Janela de 2 dias para visualizar o efeito da limpeza
t0 = clean["timestamp"].iloc[0] + pd.Timedelta(days=2)
w = clean[(clean["timestamp"] >= t0) & (clean["timestamp"] < t0 + pd.Timedelta(days=2))]
r = raw[(raw["timestamp"] >= t0) & (raw["timestamp"] < t0 + pd.Timedelta(days=2))]
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(r["timestamp"], r["libre_gl"], ".", ms=3, label="Libre (cru)")
ax.plot(r["timestamp"], r["dexcom_gl"], ".", ms=3, label="Dexcom (cru)")
ax.plot(w["timestamp"], w["glucose"], "k-", lw=1, label="glucose unificada (limpa)")
for _, m in w[w["meal_type"].notna()].iterrows():
    ax.axvline(m["timestamp"], c="orange", alpha=.6)
ax.set_ylabel("mg/dL"); ax.set_title(f"Participante {pid}: efeito da limpeza (linhas laranja = refeições)"); ax.legend(); plt.show()

## 2. Limpeza de todos os participantes + relatório de qualidade

In [ ]:
minutes, all_stats = [], []
for pid, f in files.items():
    m, s = P.clean_participant(P.load_participant_csv(f), pid, glucose_source=GLUCOSE_SOURCE, day_mode=DAY_MODE,
                               min_coverage=MIN_COVERAGE)
    minutes.append(m); all_stats.append(s)
minute = pd.concat(minutes, ignore_index=True)
quality = pd.DataFrame(all_stats)
print(minute.shape)
quality[["participant_id", "raw_rows", "duplicate_timestamps", "libre_gl_out_of_range", "dexcom_gl_out_of_range",
         "glucose_pct_missing", "hr_pct_missing", "mets_pct_missing", "dexcom_minus_libre_median", "n_meals",
         "meal_values_capped", "meal_values_imputed", "minutes_before_D1", "n_days"]].describe().T.round(2)

In [ ]:
# De onde vieram os METs de cada participante? (METs do arquivo, estimados pelas calorias, ou ausentes)
print("Fonte dos METs:", quality["mets_source"].value_counts().to_dict())
print("Participantes com METs estimados:", quality.loc[quality["mets_source"] == "estimated_from_calories", "participant_id"].tolist())
print("Participantes sem nenhuma informação de atividade:", quality.loc[quality["mets_source"] == "missing", "participant_id"].tolist())
# Macros acima dos tetos (erros de digitação), por nutriente: quem são e quantos valores
capped_cols = [c for c in quality.columns if c.startswith("capped_")]
quality.loc[quality[capped_cols].sum(axis=1) > 0, ["participant_id", *capped_cols]]

In [ ]:
quality.sort_values("glucose_pct_missing", ascending=False)[["participant_id", "glucose_pct_missing", "hr_pct_missing", "n_days"]].head(8)

## 3. Features por dia (participante × dia)
Duas famílias de variáveis:
- **Comportamento** (`BEHAVIOR_FEATURES`): o que a pessoa *faz* — macros totais e por refeição, horário das refeições, FC média, METs, minutos ativos. É isso que o algoritmo genético vai controlar em D7–D9.
- **Resposta glicêmica** (`GLUCOSE_FEATURES`): média, desvio, mín/máx, CV, TIR (% do tempo em 70–180), TAR (>180) e TBR (<70).

In [ ]:
daily = P.build_daily_features(minute, MIN_COVERAGE)
print(daily.shape)
daily.head()

In [ ]:
# Cobertura de glicose por dia: mostra quais dias são parciais
cov = daily.pivot(index="participant_id", columns="day", values="glucose_coverage")
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cov.values, aspect="auto", vmin=0, vmax=1, cmap="viridis")
ax.set_xticks(range(cov.shape[1])); ax.set_xticklabels([f"D{d}" for d in cov.columns])
ax.set_yticks(range(0, len(cov), 5)); ax.set_yticklabels(cov.index[::5]); ax.grid(False)
plt.colorbar(im, label="fração do dia com glicose"); ax.set_title("Cobertura por participante e dia"); plt.show()
print("Fração de dias completos por dia:"); print(daily.groupby("day")["day_complete"].mean().round(2).to_dict())

## 4. Separação: entrada (D1–D6) × avaliação (D7–D9)

In [ ]:
inp, ev = P.split_input_eval(daily)
print("Entrada D1-D6:", inp.shape, "| Avaliação D7-D9:", ev.shape)
daily.groupby("day")[["gl_mean", "tir", "cal_total", "carbs_total", "active_min"]].mean().round(1)

## 5. Refeições e resposta pós-prandial

In [ ]:
meals = P.extract_meals(minute)
print(meals.shape)
meals.head()

In [ ]:
ok = meals.dropna(subset=["delta_peak"])
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].scatter(ok["carbs"], ok["delta_peak"], s=5, alpha=.4); ax[0].set_xlabel("carboidratos (g)"); ax[0].set_ylabel("Δ pico de glicose (mg/dL)")
ax[0].set_title(f"Carbo × pico pós-prandial (r = {ok['carbs'].corr(ok['delta_peak']):.2f})")
ok.groupby("meal_type")["delta_peak"].mean().plot(kind="bar", ax=ax[1]); ax[1].set_title("Δ pico médio por tipo de refeição")
plt.show()

## 6. Dados dos participantes, grupos e elegibilidade
Um participante é **elegível** se tem D1…D9 e o D9 tem glicose suficiente (o D9 é o alvo do projeto).

In [ ]:
bio_path = P.find_bio_file(RAW_DIR)
bio = P.load_bio(bio_path)
participants = P.build_participants_table(daily, quality, bio, MIN_COVERAGE)
print("Elegíveis:", int(participants["eligible"].sum()), "de", len(participants))
display(participants.loc[~participants["eligible"], ["participant_id", "n_days", "reason_ineligible"]])
participants["group_a1c"].value_counts()

In [ ]:
# Conferência dos sensores com as glicemias de ponta de dedo do bio.csv (detalhes no notebook 01, seção 10)
fs_detail, fs_summary = P.fingerstick_validation(minute, bio)
print(f"{len(fs_detail)} medições com os 2 sensores | fonte usada no pipeline: {GLUCOSE_SOURCE}")
fs_summary.round(1)

## 7. Tabelas para modelagem
- **`wide`** — formulação exata do projeto: 1 linha por participante = resumo de D1–D6 + bio + comportamento de D7, D8, D9 → glicose média em D9.
  Tem só ~40 linhas (1 por participante); é a tabela que o AG usará na prática.
- **`day_level`** — ~8 linhas por participante: comportamento do dia *t* + resumo do dia *t−1* + bio → glicose do dia *t*.
  Mais linhas para treinar a MLP; o Integrante 2 pode treinar aqui e depois usar na formulação D7→D9.

In [ ]:
wide = P.build_wide_dataset(daily, participants)
day_level = P.build_day_level_dataset(daily, participants, MIN_COVERAGE)
print("wide:", wide.shape, "| day_level:", day_level.shape)
wide[["participant_id", "hist_gl_mean", "hist_carbs_total", "d9_carbs_total", "d9_active_min", "target_d9_gl_mean"]].head()

In [ ]:
# Quais variáveis mais se relacionam com o alvo? (análogo ao corr_matrix do notebook Data-Centric)
num = wide.select_dtypes("number").drop(columns=["participant_id"])
corr = num.corr()["target_d9_gl_mean"].drop(["target_d9_gl_mean", "target_d9_tir", "target_d9_gl_max"]).dropna()
print(corr.reindex(corr.abs().sort_values(ascending=False).index).head(15).round(2))

## 8. Divisão treino / validação / teste — **por participante**
Dividir por *linha* colocaria dias do mesmo participante em treino e teste (vazamento). Como na aula (`stratify=`), mantemos a proporção dos grupos clínicos nos três conjuntos.

In [ ]:
splits = P.split_participants(participants, valid_size=0.2, test_size=0.2, seed=SEED)
grp = participants.set_index("participant_id")["group_a1c"]
pd.DataFrame({k: grp.loc[v].value_counts(normalize=True).round(2) for k, v in splits.items()}).fillna(0)

## 9. Escala e imputação (sem vazamento)
Imputação pela **mediana do treino** (ex.: horário do café quando a pessoa pulou a refeição) e padronização `(x − média)/desvio` ajustada **só no treino**.
O objeto `FittedPreprocessor` também transforma dados novos — o algoritmo genético vai usá-lo para escalar os planos candidatos.

**Colunas constantes no treino são descartadas.** No dataset real, café da manhã e parte do almoço seguem o protocolo do estudo (shakes e pedidos de um mesmo restaurante) e por isso têm o mesmo valor para todos os participantes de treino em vários dias. O scaler deixaria essas colunas em unidade bruta (escala 1); então elas saem do modelo e ficam registradas em `prep.meta`.

In [ ]:
prep = P.fit_preprocessor(wide, splits, scaler="standard")
mats = P.make_matrices(wide, "target_d9_gl_mean", splits, prep)
print(len(prep.feature_cols), "features |", {k: v.shape for k, v in mats.items() if k.startswith("X_")})
print(f"Descartadas por serem constantes no treino ({len(prep.meta['dropped_constant'])}):", prep.meta["dropped_constant"])
print("Média das features no treino (≈0):", round(float(mats["X_train"].mean().abs().max()), 6))
print("Valores faltantes restantes em X_train/valid/test:", [int(mats[f"X_{s}"].isna().sum().sum()) for s in ("train", "valid", "test")])

## 10. Salvar a base organizada
`run_pipeline` repete todas as etapas acima e grava os arquivos em `data/processed/` (descritos no README). Rode esta célula e avise os Integrantes 2 e 3.

In [ ]:
res = P.run_pipeline(RAW_DIR, OUT_DIR, bio_path=bio_path, glucose_source=GLUCOSE_SOURCE, day_mode=DAY_MODE,
                     min_coverage=MIN_COVERAGE, seed=SEED)
print(sorted(p.name for p in OUT_DIR.iterdir()))

In [ ]:
# Conferência: como os outros integrantes vão carregar
data = P.load_processed(OUT_DIR)
prep = P.load_preprocessor(OUT_DIR / "model_ready" / "wide" / "preprocessor.pkl")
X_train = pd.read_csv(OUT_DIR / "model_ready" / "wide" / "X_train.csv")
y_train = pd.read_csv(OUT_DIR / "model_ready" / "wide" / "y_train.csv")
print(X_train.shape, y_train.shape, "| exemplo de feature:", prep.feature_cols[:3])